# CampaignIQ — Statistical Validation Report

This notebook validates the production A/B-testing statistical engine against the **Cookie Cats** experiment used in the Marketing Campaign Optimizer project.

### Validation objectives
- Compare Gate 30 vs Gate 40 for Day-1 retention.
- Compare Gate 30 vs Gate 40 for Day-7 retention.
- Calculate group rates, absolute differences, z-statistics, p-values, and 95% confidence intervals.
- Verify that the production `significance_test.py` produces consistent results.
- Demonstrate the correct interpretation of statistically significant and non-significant results.


## 1. Project setup

Run this notebook in **Jupyter Notebook in your browser** from the project environment. The notebook uses the project's processed Cookie Cats dataset and production statistical engine.

In [16]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

PROJECT_ROOT = Path.cwd()

# If Jupyter was launched from the notebooks folder, move to the project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cookie_cats_processed.csv"
SCRIPTS_PATH = PROJECT_ROOT / "scripts"

if str(SCRIPTS_PATH) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_PATH))

from significance_test import two_proportion_test

ALPHA = 0.05
DATA_PATH

WindowsPath('E:/Thanuj_V/Projects/marketing_campaign_optimizer/data/processed/cookie_cats_processed.csv')

## 2. Load and validate the processed Cookie Cats data

In [17]:
cookie = pd.read_csv(DATA_PATH)

print(f"Shape: {cookie.shape}")
print("Columns:", cookie.columns.tolist())
            
display(cookie.head())
assert cookie.shape[0] > 0
assert {"userid", "version", "retention_1", "retention_7"}.issubset(cookie.columns)
assert cookie["userid"].is_unique
assert cookie[["retention_1", "retention_7"]].isna().sum().sum() == 0
assert set(cookie["version"].unique()) == {"gate_30", "gate_40"}
assert cookie["retention_1"].isin([0, 1, True, False]).all()
assert cookie["retention_7"].isin([0, 1, True, False]).all()

print("\nValidation checks: PASSED")

Shape: (90189, 5)
Columns: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True



Validation checks: PASSED


## 3. Experiment overview

The experiment compares users assigned to **gate_30** and **gate_40**. We treat gate_30 as the control group and gate_40 as the treatment group for this validation.

In [18]:
overview = (
    cookie.groupby("version")
    .agg(
        users=("userid", "count"),
        retention_1=("retention_1", "mean"),
        retention_7=("retention_7", "mean"),
    )
    .reset_index()
)

overview["retention_1"] = overview["retention_1"].map(lambda x: f"{x:.2%}")
overview["retention_7"] = overview["retention_7"].map(lambda x: f"{x:.2%}")

display(overview)

,version,users,retention_1,retention_7
0,gate_30,44700,44.82%,19.02%
1,gate_40,45489,44.23%,18.20%


## 4. Validation helper

The helper below performs the same two-proportion z-test and confidence-interval calculation used by the project. The confidence interval is for **treatment − control**.

In [19]:
def validate_proportion_metric(df, metric, control_value="gate_30", treatment_value="gate_40", alpha=0.05):
    control = df.loc[df["version"] == control_value, metric].astype(int)
    treatment = df.loc[df["version"] == treatment_value, metric].astype(int)

    control_n = len(control)
    treatment_n = len(treatment)
    control_successes = int(control.sum())
    treatment_successes = int(treatment.sum())

    control_rate = control_successes / control_n
    treatment_rate = treatment_successes / treatment_n
    difference = treatment_rate - control_rate
    relative_lift = difference / control_rate

    count = np.array([treatment_successes, control_successes])
    nobs = np.array([treatment_n, control_n])

    z_stat, p_value = proportions_ztest(count, nobs)

    ci_low, ci_high = confint_proportions_2indep(
        treatment_successes,
        treatment_n,
        control_successes,
        control_n,
        method="wald",
        compare="diff",
        alpha=alpha,
    )

    return {
        "metric": metric,
        "control_n": control_n,
        "treatment_n": treatment_n,
        "control_successes": control_successes,
        "treatment_successes": treatment_successes,
        "control_rate": control_rate,
        "treatment_rate": treatment_rate,
        "absolute_difference": difference,
        "relative_lift": relative_lift,
        "z_stat": z_stat,
        "p_value": p_value,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "statistically_significant": p_value < alpha,
    }


## 5. Day-1 retention validation

In [20]:
day1 = validate_proportion_metric(cookie, "retention_1", alpha=ALPHA)

print(f"Control rate:      {day1['control_rate']:.6%}")
print(f"Treatment rate:    {day1['treatment_rate']:.6%}")
print(f"Difference:        {day1['absolute_difference']:.6%}")
print(f"Relative lift:     {day1['relative_lift']:.2%}")
print(f"z-statistic:       {day1['z_stat']:.4f}")
print(f"p-value:           {day1['p_value']:.10f}")
print(f"95% CI:            [{day1['ci_low']:.6%}, {day1['ci_high']:.6%}]")
print(f"Significant:       {day1['statistically_significant']}")

Control rate:      44.818792%
Treatment rate:    44.228275%
Difference:        -0.590517%
Relative lift:     -1.32%
z-statistic:       -1.7841
p-value:           0.0744096553
95% CI:            [-1.239244%, 0.058210%]
Significant:       False


### Day-1 interpretation

The expected validation outcome is that Day-1 retention is **not statistically significant at α = 0.05**. A non-significant result does **not** prove that the two variants are identical; it means the analysis did not provide sufficient statistical evidence of a difference under the selected test and threshold.

In [21]:
assert day1["p_value"] > ALPHA
print("Day-1 validation verdict: PASS — fail to reject the null hypothesis at α = 0.05.")

Day-1 validation verdict: PASS — fail to reject the null hypothesis at α = 0.05.


## 6. Day-7 retention validation

In [22]:
day7 = validate_proportion_metric(cookie, "retention_7", alpha=ALPHA)

print(f"Control rate:      {day7['control_rate']:.6%}")
print(f"Treatment rate:    {day7['treatment_rate']:.6%}")
print(f"Difference:        {day7['absolute_difference']:.6%}")
print(f"Relative lift:     {day7['relative_lift']:.2%}")
print(f"z-statistic:       {day7['z_stat']:.4f}")
print(f"p-value:           {day7['p_value']:.10f}")
print(f"95% CI:            [{day7['ci_low']:.6%}, {day7['ci_high']:.6%}]")
print(f"Significant:       {day7['statistically_significant']}")

Control rate:      19.020134%
Treatment rate:    18.200004%
Difference:        -0.820130%
Relative lift:     -4.31%
z-statistic:       -3.1644
p-value:           0.0015542500
95% CI:            [-1.328155%, -0.312104%]
Significant:       True


### Day-7 interpretation

The expected validation outcome is that Day-7 retention is **statistically significant at α = 0.05**, with the treatment group showing a lower retention rate than the control group.

In [23]:
assert day7["p_value"] < ALPHA
assert day7["absolute_difference"] < 0
print("Day-7 validation verdict: PASS — statistically significant difference detected at α = 0.05.")

Day-7 validation verdict: PASS — statistically significant difference detected at α = 0.05.


## 7. Compare both validation results

In [24]:
validation_summary = pd.DataFrame([
    {
        "Metric": "Day-1 retention",
        "Control": day1["control_rate"],
        "Treatment": day1["treatment_rate"],
        "Difference": day1["absolute_difference"],
        "Relative lift": day1["relative_lift"],
        "z": day1["z_stat"],
        "p-value": day1["p_value"],
        "CI low": day1["ci_low"],
        "CI high": day1["ci_high"],
        "Significant": day1["statistically_significant"],
    },
    {
        "Metric": "Day-7 retention",
        "Control": day7["control_rate"],
        "Treatment": day7["treatment_rate"],
        "Difference": day7["absolute_difference"],
        "Relative lift": day7["relative_lift"],
        "z": day7["z_stat"],
        "p-value": day7["p_value"],
        "CI low": day7["ci_low"],
        "CI high": day7["ci_high"],
        "Significant": day7["statistically_significant"],
    },
])

display(validation_summary.style.format({
    "Control": "{:.2%}",
    "Treatment": "{:.2%}",
    "Difference": "{:.2%}",
    "Relative lift": "{:.2%}",
    "z": "{:.4f}",
    "p-value": "{:.6f}",
    "CI low": "{:.2%}",
    "CI high": "{:.2%}",
}))

,Metric,Control,Treatment,Difference,Relative lift,z,p-value,CI low,CI high,Significant
0,Day-1 retention,44.82%,44.23%,-0.59%,-1.32%,-1.7841,0.074410,-1.24%,0.06%,False
1,Day-7 retention,19.02%,18.20%,-0.82%,-4.31%,-3.1644,0.001554,-1.33%,-0.31%,True


## 8. Validate the production `significance_test.py`

The production function is called with the project's existing positional argument order. This check confirms that the production engine agrees with the notebook calculation on the key statistical outputs.

In [25]:
def production_result(metric_result):
    return two_proportion_test(
        metric_result["control_successes"],
        metric_result["control_n"],
        metric_result["treatment_successes"],
        metric_result["treatment_n"],
        ALPHA,
    )

prod_day1 = production_result(day1)
prod_day7 = production_result(day7)

print("Production Day-1 result:")
print(prod_day1)
print("\nProduction Day-7 result:")
print(prod_day7)

Production Day-1 result:
{'treatment_rate': 0.4481879194630872, 'control_rate': 0.44228274967574577, 'observed_difference': 0.005905169787341458, 'relative_lift': 0.013351571571965584, 'z_statistic': np.float64(1.7840862247974725), 'p_value': np.float64(0.07440965529691913), 'alpha': 0.05, 'ci_low': np.float64(-0.0005820998747623034), 'ci_high': np.float64(0.012392439449445219), 'statistically_significant': np.False_, 'decision': 'Fail to reject H0'}

Production Day-7 result:
{'treatment_rate': 0.19020134228187918, 'control_rate': 0.18200004396667327, 'observed_difference': 0.008201298315205913, 'relative_lift': 0.04506206776910276, 'z_statistic': np.float64(3.164358912748191), 'p_value': np.float64(0.001554249975614329), 'alpha': 0.05, 'ci_low': np.float64(0.00312104421152628), 'ci_high': np.float64(0.013281552418885546), 'statistically_significant': np.True_, 'decision': 'Reject H0'}


In [28]:

# =========================================================
# PRODUCTION ENGINE VALIDATION
# =========================================================

def run_production_validation(metric_result):
    """
    Run the project's production significance_test.py.

    IMPORTANT:
    The production function expects:
        treatment_successes,
        treatment_total,
        control_successes,
        control_total,
        alpha
    """

    return two_proportion_test(
        metric_result["treatment_successes"],
        metric_result["treatment_n"],
        metric_result["control_successes"],
        metric_result["control_n"],
        ALPHA,
    )


# ---------------------------------------------------------
# Recalculate production results from scratch
# ---------------------------------------------------------

prod_day1 = run_production_validation(day1)
prod_day7 = run_production_validation(day7)


# ---------------------------------------------------------
# Validation function
# ---------------------------------------------------------

def check_production_match(metric_name, notebook_result, production_result):

    checks = {
        "control_rate": (
            notebook_result["control_rate"],
            production_result["control_rate"],
        ),
        "treatment_rate": (
            notebook_result["treatment_rate"],
            production_result["treatment_rate"],
        ),
        "p_value": (
            notebook_result["p_value"],
            production_result["p_value"],
        ),
        "ci_low": (
            notebook_result["ci_low"],
            production_result["ci_low"],
        ),
        "ci_high": (
            notebook_result["ci_high"],
            production_result["ci_high"],
        ),
    }

    print("\n" + "=" * 70)
    print(f"{metric_name.upper()} — PRODUCTION ENGINE VALIDATION")
    print("=" * 70)

    all_passed = True

    for key, (notebook_value, production_value) in checks.items():

        passed = np.isclose(
            notebook_value,
            production_value,
            rtol=1e-8,
            atol=1e-10,
        )

        print(f"\n{key}")
        print(f"  Notebook:   {notebook_value:.10f}")
        print(f"  Production: {production_value:.10f}")

        if passed:
            print("  Status:     PASS")
        else:
            print("  Status:     FAIL")
            all_passed = False

    print("\n" + "-" * 70)

    if all_passed:
        print(f"{metric_name} validation: PASS")
    else:
        print(f"{metric_name} validation: FAIL")

    return all_passed


# ---------------------------------------------------------
# Validate Day-1
# ---------------------------------------------------------

day1_passed = check_production_match(
    "Day-1 retention",
    day1,
    prod_day1,
)


# ---------------------------------------------------------
# Validate Day-7
# ---------------------------------------------------------

day7_passed = check_production_match(
    "Day-7 retention",
    day7,
    prod_day7,
)


# ---------------------------------------------------------
# Final result
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL PRODUCTION ENGINE VALIDATION")
print("=" * 70)

if day1_passed and day7_passed:
    print("✓ Day-1 production validation: PASS")
    print("✓ Day-7 production validation: PASS")
    print("✓ Production engine validation: PASS")
    print("\nThe notebook and production statistical engine agree.")
else:
    print("✗ Production engine validation: FAIL")
    print("\nReview the mismatch values printed above.")



DAY-1 RETENTION — PRODUCTION ENGINE VALIDATION

control_rate
  Notebook:   0.4481879195
  Production: 0.4481879195
  Status:     PASS

treatment_rate
  Notebook:   0.4422827497
  Production: 0.4422827497
  Status:     PASS

p_value
  Notebook:   0.0744096553
  Production: 0.0744096553
  Status:     PASS

ci_low
  Notebook:   -0.0123924394
  Production: -0.0123924394
  Status:     PASS

ci_high
  Notebook:   0.0005820999
  Production: 0.0005820999
  Status:     PASS

----------------------------------------------------------------------
Day-1 retention validation: PASS

DAY-7 RETENTION — PRODUCTION ENGINE VALIDATION

control_rate
  Notebook:   0.1902013423
  Production: 0.1902013423
  Status:     PASS

treatment_rate
  Notebook:   0.1820000440
  Production: 0.1820000440
  Status:     PASS

p_value
  Notebook:   0.0015542500
  Production: 0.0015542500
  Status:     PASS

ci_low
  Notebook:   -0.0132815524
  Production: -0.0132815524
  Status:     PASS

ci_high
  Notebook:   -0.003121044

## 9. Final validation conclusions

### Results supported by this notebook
- **Day-1 retention:** no statistically significant difference detected at α = 0.05.
- **Day-7 retention:** statistically significant difference detected at α = 0.05, with lower treatment retention.
- The production `significance_test.py` agrees with the notebook calculations for the validated metrics.
- Statistical non-significance is not interpreted as proof that the variants are identical.

### Important statistical note
The confidence interval describes plausible values for the treatment-minus-control difference under the selected interval method. Sample size and statistical power should also be considered before making an experimentation decision.